# Inspect model adapters

Loads the current foundation-model adapters and checks their output on benchmark windows.

In [ ]:
import pandas as pd
from torch.utils.data import Subset

from eeg_benchmark.adapters import adapt_windows_to_model
from eeg_benchmark.datasets import MOABBDatasetWrapper, split_train_test
from eeg_benchmark.models import MODELS
from eeg_benchmark.pipeline import encode_dataset

In [ ]:
dataset = MOABBDatasetWrapper.load("BNCI2014_001")

In [ ]:
rows = []

for model_name, model_cls in MODELS.items():
    model = model_cls.load(device="cpu")
    windows = adapt_windows_to_model(dataset.create_event_windows(), model)
    train_windows, _ = split_train_test(windows)
    embeddings, labels = encode_dataset(
        Subset(train_windows, range(2)),
        model,
        dataset.channel_names,
        batch_size=2,
    )

    rows.append(
        {
            "model": model_name,
            "backbone": model.model.__class__.__name__,
            "parameters": sum(parameter.numel() for parameter in model.model.parameters()),
            "input_window_samples": model.input_window_samples,
            "embedding_shape": "x".join(str(size) for size in embeddings.shape),
            "labels": labels.tolist(),
        }
    )

pd.DataFrame(rows)